In [190]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

column_names = ['ep (ms)', 'Acc_x', 'Acc_y', 'Acc_z', 'Gyro_x', 'Gyro_y', 'Gyro_z', 'ID', 'Label', 'Category', 'Set']
to_delete = ['ep (ms)', 'Acc_x', 'Acc_y', 'Acc_z', 'Gyro_x', 'Gyro_y', 'Gyro_z']

# Read the CSV while specifying column names
df = pd.read_csv('data2.csv', names=column_names, skiprows=1,sep=',')
df = df.apply(lambda column: column.fillna(column.mode()[0]))
df = df.drop(columns=to_delete)

In [191]:
df

,ID,Label,Category,Set
0,B,bench,heavy,30.0
1,B,bench,heavy,30.0
2,B,bench,heavy,30.0
3,B,bench,heavy,30.0
4,B,bench,heavy,30.0
...,...,...,...,...
9004,E,row,medium,40.0
9005,E,row,medium,40.0
9006,E,row,medium,40.0
9007,E,row,medium,40.0


In [192]:
df['items'] = df[['Label', 'Category']].apply(lambda x: '_'.join(x.astype(str)), axis=1)
df

,ID,Label,Category,Set,items
0,B,bench,heavy,30.0,bench_heavy
1,B,bench,heavy,30.0,bench_heavy
2,B,bench,heavy,30.0,bench_heavy
3,B,bench,heavy,30.0,bench_heavy
4,B,bench,heavy,30.0,bench_heavy
...,...,...,...,...,...
9004,E,row,medium,40.0,row_medium
9005,E,row,medium,40.0,row_medium
9006,E,row,medium,40.0,row_medium
9007,E,row,medium,40.0,row_medium


In [193]:
df_items = df[["ID", "items"]]
df_items

,ID,items
0,B,bench_heavy
1,B,bench_heavy
2,B,bench_heavy
3,B,bench_heavy
4,B,bench_heavy
...,...,...
9004,E,row_medium
9005,E,row_medium
9006,E,row_medium
9007,E,row_medium


In [194]:
df_items = df_items.drop_duplicates()
df_items

,ID,items
0,B,bench_heavy
15,B,bench_medium
85,A,bench_heavy
312,A,ohp_heavy
388,B,ohp_heavy
776,B,ohp_medium
885,A,ohp_medium
1380,A,squat_medium
1461,B,squat_medium
1843,A,dead_medium


In [195]:
df_items = df_items.groupby('ID')['items'].apply(list).reset_index()

df_items


,ID,items
0,A,"[bench_heavy, ohp_heavy, ohp_medium, squat_med..."
1,B,"[bench_heavy, bench_medium, ohp_heavy, ohp_med..."
2,C,"[bench_heavy, ohp_heavy, ohp_heav, row_medium,..."
3,D,"[squat_medium, squat_heavy, bench_medium, row_..."
4,E,"[bench_heavy, dead_medium, ohp_heavy, row_heav..."


In [196]:
df_items['item_count'] = df_items['items'].apply(len)
df_items

,ID,items,item_count
0,A,"[bench_heavy, ohp_heavy, ohp_medium, squat_med...",11
1,B,"[bench_heavy, bench_medium, ohp_heavy, ohp_med...",5
2,C,"[bench_heavy, ohp_heavy, ohp_heav, row_medium,...",8
3,D,"[squat_medium, squat_heavy, bench_medium, row_...",5
4,E,"[bench_heavy, dead_medium, ohp_heavy, row_heav...",10


In [197]:
number_of_items = df_items['item_count'].sum()
number_of_items
print("Number of transactions: ", len(df_items))
print("Number of items: ", number_of_items)



Number of transactions:  5
Number of items:  39


In [198]:
df_items.to_csv('DataExos_2.csv', index=False)

In [199]:
from mlxtend.frequent_patterns import apriori, association_rules


In [200]:
transactions = df_items['items'].tolist()

# Display transactions (optional)
print(transactions)

[['bench_heavy', 'ohp_heavy', 'ohp_medium', 'squat_medium', 'dead_medium', 'row_heavy', 'squat_heavy', 'dead_heavy', 'rest_sitting', 'reste_sitting', 'rest_standing'], ['bench_heavy', 'bench_medium', 'ohp_heavy', 'ohp_medium', 'squat_medium'], ['bench_heavy', 'ohp_heavy', 'ohp_heav', 'row_medium', 'row_heavy', 'raw_heavy', 'squat_heavy', 'dead_medium'], ['squat_medium', 'squat_heavy', 'bench_medium', 'row_medium', 'raw_medium'], ['bench_heavy', 'dead_medium', 'ohp_heavy', 'row_heavy', 'squat_heavy', 'dead_heavy', 'bench_medium', 'row_medium', 'rest_sitting', 'rest_standing']]


In [201]:
from mlxtend.preprocessing import TransactionEncoder

# Use TransactionEncoder to one-hot encode the transactions
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)

# Create a DataFrame
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)
df_encoded

,bench_heavy,bench_medium,dead_heavy,dead_medium,ohp_heav,ohp_heavy,ohp_medium,raw_heavy,raw_medium,rest_sitting,rest_standing,reste_sitting,row_heavy,row_medium,squat_heavy,squat_medium
0,True,False,True,True,False,True,True,False,False,True,True,True,True,False,True,True
1,True,True,False,False,False,True,True,False,False,False,False,False,False,False,False,True
2,True,False,False,True,True,True,False,True,False,False,False,False,True,True,True,False
3,False,True,False,False,False,False,False,False,True,False,False,False,False,True,True,True
4,True,True,True,True,False,True,False,False,False,True,True,False,True,True,True,False


In [237]:
# Apply Apriori to find frequent itemsets with minimum support
frequent_itemsets = apriori(df_encoded, min_support=0.6, use_colnames=True)

# Display frequent itemsets
print(frequent_itemsets)


    support                                           itemsets
0       0.8                                      (bench_heavy)
1       0.6                                     (bench_medium)
2       0.6                                      (dead_medium)
3       0.8                                        (ohp_heavy)
4       0.6                                        (row_heavy)
5       0.6                                       (row_medium)
6       0.8                                      (squat_heavy)
7       0.6                                     (squat_medium)
8       0.6                         (dead_medium, bench_heavy)
9       0.8                           (ohp_heavy, bench_heavy)
10      0.6                           (bench_heavy, row_heavy)
11      0.6                         (bench_heavy, squat_heavy)
12      0.6                           (dead_medium, ohp_heavy)
13      0.6                           (dead_medium, row_heavy)
14      0.6                         (dead_medium, squat

In [238]:
print(frequent_itemsets)
frequent_itemsets.to_csv('FrequentItemsets_2.csv', index=False)

    support                                           itemsets
0       0.8                                      (bench_heavy)
1       0.6                                     (bench_medium)
2       0.6                                      (dead_medium)
3       0.8                                        (ohp_heavy)
4       0.6                                        (row_heavy)
5       0.6                                       (row_medium)
6       0.8                                      (squat_heavy)
7       0.6                                     (squat_medium)
8       0.6                         (dead_medium, bench_heavy)
9       0.8                           (ohp_heavy, bench_heavy)
10      0.6                           (bench_heavy, row_heavy)
11      0.6                         (bench_heavy, squat_heavy)
12      0.6                           (dead_medium, ohp_heavy)
13      0.6                           (dead_medium, row_heavy)
14      0.6                         (dead_medium, squat

In [204]:
# Generate association rules with minimum confidence
rules = association_rules(frequent_itemsets,num_itemsets=len(df_encoded), metric="confidence", min_threshold=0.7)


# Display the rules
print(rules)


       antecedents                                        consequents  \
0    (dead_medium)                                      (bench_heavy)   
1    (bench_heavy)                                      (dead_medium)   
2      (ohp_heavy)                                      (bench_heavy)   
3    (bench_heavy)                                        (ohp_heavy)   
4    (bench_heavy)                                        (row_heavy)   
..             ...                                                ...   
177  (squat_heavy)   (ohp_heavy, bench_heavy, dead_medium, row_heavy)   
178    (ohp_heavy)  (dead_medium, bench_heavy, squat_heavy, row_he...   
179    (row_heavy)  (ohp_heavy, bench_heavy, squat_heavy, dead_med...   
180  (dead_medium)   (ohp_heavy, bench_heavy, squat_heavy, row_heavy)   
181  (bench_heavy)   (ohp_heavy, squat_heavy, dead_medium, row_heavy)   

     antecedent support  consequent support  support  confidence      lift  \
0                   0.6                 0.8  

In [ ]:
def generate_candidates(L_prev, k):
    candidates = []

    for i in range(len(L_prev)):
        for j in range(i + 1, len(L_prev)):
            itemset1 = list(L_prev[i])
            itemset2 = list(L_prev[j])

            # Compare first (k-1) elements
            if itemset1[:-1] == itemset2[:-1]:  
                candidate = L_prev[i].union(L_prev[j])  # Union of two itemsets to form k-itemset
                candidates.append(candidate)

    return candidates

def calculate_support(transactions, candidates):
    support = {}
    num_transactions = len(transactions)

    for candidate in candidates:
        count = sum(1 for transaction in transactions if candidate.issubset(transaction))
        support[frozenset(candidate)] = count / num_transactions

    return support
def generate_frequent_itemsets(candidates, support, min_support):
    frequent_itemsets = [itemset for itemset, supp in support.items() if supp >= min_support]
    return frequent_itemsets


In [240]:
# Prepare the data from df_items
transactions = df_items['items'].tolist()  # List of itemsets for each transaction

# Generate 1-itemsets (C1) from transactions
L_prev = [frozenset([item]) for transaction in transactions for item in transaction]
L_prev = list(set(L_prev))  # Remove duplicates

# Set minimum support
min_support = 0.6  # Example support threshold

# Example: Generate 2-itemset candidates (C2)
k = 2

while L_prev:
    frequent = [list(itemset) for itemset in L_prev]
    # print(f"Frequent {k-1}-itemsets len = {len(frequent)}: {frequent}")

    # Generate Ck
    candidates = generate_candidates(L_prev, k)
    print(f"Candidates {k}-itemsets len {len(candidates)}: {candidates}")
    # Calculate support for Ck
    support = calculate_support(transactions, candidates)
    
    # Generate Lk
    L_prev = generate_frequent_itemsets(candidates, support, min_support)
    
    if not L_prev:
        break
    
    k += 1  # Increment k for next iteration


Candidates 2-itemsets len 120: [frozenset({'dead_medium', 'reste_sitting'}), frozenset({'dead_medium', 'squat_medium'}), frozenset({'dead_medium', 'raw_heavy'}), frozenset({'dead_medium', 'ohp_medium'}), frozenset({'dead_medium', 'row_heavy'}), frozenset({'dead_medium', 'squat_heavy'}), frozenset({'dead_medium', 'ohp_heav'}), frozenset({'dead_medium', 'bench_medium'}), frozenset({'dead_medium', 'rest_standing'}), frozenset({'dead_medium', 'raw_medium'}), frozenset({'dead_medium', 'dead_heavy'}), frozenset({'dead_medium', 'rest_sitting'}), frozenset({'dead_medium', 'row_medium'}), frozenset({'dead_medium', 'bench_heavy'}), frozenset({'dead_medium', 'ohp_heavy'}), frozenset({'squat_medium', 'reste_sitting'}), frozenset({'raw_heavy', 'reste_sitting'}), frozenset({'ohp_medium', 'reste_sitting'}), frozenset({'row_heavy', 'reste_sitting'}), frozenset({'squat_heavy', 'reste_sitting'}), frozenset({'ohp_heav', 'reste_sitting'}), frozenset({'bench_medium', 'reste_sitting'}), frozenset({'rest_sta

In [207]:
from itertools import combinations

def generate_candidates(transactions):
    # Step 1: Generate 1-itemsets (C1)
    C1 = []
    for transaction in transactions:
        for item in transaction:
            C1.append(frozenset([item]))  # Store each item as a frozenset

    C1 = list(set(C1))  # Remove duplicates
    all_candidates = C1  # Start with 1-itemsets
    current_candidates = C1  # Set of candidates to start with

    size = 2
    while current_candidates:
        new_candidates = []

        # Generate candidates by merging itemsets of previous size
        for i in range(len(current_candidates)):
            for j in range(i + 1, len(current_candidates)):
                candidate = current_candidates[i].union(current_candidates[j])

                # Add to new_candidates if it's a valid candidate
                if len(candidate) == size:
                    new_candidates.append(candidate)

        # If no new candidates were found, stop the loop
        if not new_candidates:
            break

        # Add the new candidates of the current size to the list
        all_candidates.extend(new_candidates)
        current_candidates = new_candidates  # Update the candidates for the next size

        # Move to the next size
        size += 1

    return all_candidates


# Sample transactions (list of itemsets)
transactions = [
    {'apple', 'banana', 'cherry'},
    {'apple', 'banana'},
    {'banana', 'cherry'},
    {'apple', 'cherry'}
]

# Generate all candidate itemsets
candidates = generate_candidates(transactions)

# Print the generated candidates
print("Generated candidates:", candidates)


Generated candidates: [frozenset({'banana'}), frozenset({'apple'}), frozenset({'cherry'}), frozenset({'apple', 'banana'}), frozenset({'banana', 'cherry'}), frozenset({'apple', 'cherry'}), frozenset({'apple', 'banana', 'cherry'}), frozenset({'apple', 'banana', 'cherry'}), frozenset({'apple', 'banana', 'cherry'})]
